# Intersecting Regular Expressions

In the lecture notes we have shown that the intersection $L_1 \cap L_2$ of two regular languages $L_1$ and $L_2$
is again a regular language.  The proof constructs the *product automaton* $F_1 \times F_2$ of two complete
<span style="font-variant:small-caps;">Dfa</span>s $F_1$ and $F_2$ that accept $L_1$ and $L_2$.  Therefore, there is a function
$$ \wedge: \textsf{RegExp} \times \textsf{RegExp} \rightarrow \textsf{RegExp} $$
such that
$$ L(r_1 \wedge r_2) = L(r_1) \cap L(r_2) $$
holds for all regular expressions $r_1$ and $r_2$.  In this notebook we implement this function.  Given two regular
expressions $r_1$ and $r_2$, the regular expression $r_1 \wedge r_2$ is computed as follows:
1. $r_1$ and $r_2$ are converted into <span style="font-variant:small-caps;">Nfa</span>s, which are then converted into <span style="font-variant:small-caps;">Dfa</span>s $F_1$ and $F_2$.
2. The <span style="font-variant:small-caps;">Dfa</span>s $F_1$ and $F_2$ are minimized.  This step is not needed for the construction, but it keeps the
   <span style="font-variant:small-caps;">Dfa</span>s small.
3. We compute the product automaton $F_1 \times F_2$.
4. The product automaton is converted into a regular expression.

We will see that the resulting regular expression is absurdly large, even for very simple regular expressions
$r_1$ and $r_2$.  This is the reason why the function $\wedge$ is of no practical use.

The notebook `09-Equivalence.ipynb` provides the function `regexp2DFA`, which converts a regular expression into a <span style="font-variant:small-caps;">Dfa</span>, and the functions `fsm_complement` and `is_empty`, which we use to check our results.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "09-Equivalence.ipynb"

The notebook `07-Minimize.ipynb` provides the function `minimize`, which minimizes a <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
run_notebook "07-Minimize.ipynb"

The notebook `05-DFA-2-RegExp.ipynb` provides the function `dfa_2_regexp`, which converts a <span style="font-variant:small-caps;">Dfa</span> into a regular expression.

In [ ]:
run_notebook "05-DFA-2-RegExp.ipynb"

The notebook `Simplify.ipynb` provides the function `simplify`, which simplifies a regular expression, and the function `to_str`, which turns a regular expression into a string that contains only the parentheses that are needed.  The notebook `RegExp-Parser.ipynb` provides the function `parse`, which turns a string into a regular expression.

In [ ]:
run_notebook "Simplify.ipynb";;
run_notebook "RegExp-Parser.ipynb"

**Note:** The function `run_notebook` executes all cells of a notebook, just like the magic command `%run`.  The
Python version has to mark the calls of functions that are defined in the loaded notebooks with
`# type: ignore`.  The OCaml compiler knows these functions, so every call is type checked.

The regular expressions computed in this notebook are huge.  Therefore, the directive `#print_depth 4` tells the
notebook to show only the top four levels of nested values.  The function `Unix.gettimeofday`, which is used to
measure the running time, corresponds to Python's `time.perf_counter`.

In [ ]:
#print_depth 4

In [ ]:
#show_type regexp;;
#show_type dfa

Given two complete <span style="font-variant:small-caps;">Dfa</span>s
$$ F_1 = \langle Q_1, \Sigma, \delta_1, q_1, A_1 \rangle \quad\mbox{and}\quad
   F_2 = \langle Q_2, \Sigma, \delta_2, q_2, A_2 \rangle, $$
the *product automaton* $F_1 \times F_2$ is defined as
$$ F_1 \times F_2 := \bigl\langle Q_1 \times Q_2, \Sigma, \delta, \langle q_1, q_2 \rangle, A_1 \times A_2 \bigr\rangle, $$
where the transition function $\delta$ is given by
$$ \delta\bigl(\langle p_1, p_2 \rangle, c\bigr) := \bigl\langle \delta_1(p_1, c), \delta_2(p_2, c) \bigr\rangle. $$
The product automaton runs $F_1$ and $F_2$ in parallel and accepts a string iff both $F_1$ and $F_2$ accept it.

**Function `product_automaton f1 f2`**
- *Input:* `f1` and `f2` are complete <span style="font-variant:small-caps;">Dfa</span>s with the same alphabet.
- *Output:* The product automaton $F_1 \times F_2$, which accepts the language $L(F_1) \cap L(F_2)$.

**Type safety:** The states of `f1` and `f2` may have different types `'p` and `'q`.  The states of the product
automaton are pairs of type `'p * 'q`.  The Python version needs `# type: ignore` here, since its types are not
precise enough.

In [ ]:
let product_automaton (f1 : 'p dfa) (f2 : 'q dfa) : ('p * 'q) dfa =
  let ({ states = states1; sigma; delta = delta1; start = q1; accepting = a1 } : 'p dfa) = f1 in
  let ({ states = states2; delta = delta2; start = q2; accepting = a2; _ } : 'q dfa) = f2 in
  let states = cartesian_product states1 states2 in
  let delta = ref Dict.empty in
  FSet.iter
    (fun (p1, p2) ->
      FSet.iter (fun c -> delta := Dict.add ((p1, p2), c) (Dict.find (p1, c) delta1, Dict.find (p2, c) delta2) !delta) sigma)
    states;
  { states; sigma; delta = !delta; start = (q1, q2); accepting = cartesian_product a1 a2 }

**Function `intersection_dfa r1 r2 sigma`**
- *Input:* `r1` and `r2` are regular expressions and `sigma` is the alphabet used in them.
- *Output:* The product automaton of the minimized <span style="font-variant:small-caps;">Dfa</span>s for `r1` and `r2`.  It accepts the language
  $L(r_1) \cap L(r_2)$.

In [ ]:
let intersection_dfa (r1 : regexp) (r2 : regexp) (sigma : char FSet.t) : (state FSet.t set_state * state FSet.t set_state) dfa =
  let f1 = minimize (regexp2DFA r1 sigma) in
  let f2 = minimize (regexp2DFA r2 sigma) in
  product_automaton f1 f2

**Function `intersection r1 r2 sigma`**
- *Input:* `r1` and `r2` are regular expressions and `sigma` is the alphabet used in them.
- *Output:* The regular expression $r_1 \wedge r_2$, which satisfies $L(r_1 \wedge r_2) = L(r_1) \cap L(r_2)$.
  It is not simplified.

In [ ]:
let intersection (r1 : regexp) (r2 : regexp) (sigma : char FSet.t) : regexp =
  dfa_2_regexp (intersection_dfa r1 r2 sigma)

To measure the size of a regular expression we count the nodes of its syntax tree:  Every character, every
occurrence of $\varepsilon$ and $\emptyset$, and every operator counts as one node.

**Function `size r`**
- *Input:* `r` is a regular expression, represented as a value of type `regexp`.
- *Output:* The number of nodes of the syntax tree of `r`.

In [ ]:
let rec size (r : regexp) : int =
  match r with
  | Star r1 -> 1 + size r1
  | Sum (r1, r2) | Product (r1, r2) -> 1 + size r1 + size r2
  | Empty | Epsilon | Char _ -> 1

The function `accepts_same_language` checks whether the <span style="font-variant:small-caps;">Dfa</span> computed from a regular expression `r` accepts
the same language as the <span style="font-variant:small-caps;">Dfa</span> `f`.  We use it to check that the regular expressions computed by
`intersection` are correct.

**Function `accepts_same_language r f sigma`**
- *Input:* `r` is a regular expression, `f` is a complete <span style="font-variant:small-caps;">Dfa</span>, and `sigma` is its alphabet.
- *Output:* `true` if $L(r) = L(F)$, `false` otherwise.

In [ ]:
let accepts_same_language (r : regexp) (f : 'q dfa) (sigma : char FSet.t) : bool =
  let d = regexp2DFA r sigma in
  is_empty (fsm_complement d f) && is_empty (fsm_complement f d)

## A First Example

The regular expression $r_1$ describes the strings over the alphabet $\{\texttt{a}, \texttt{b}\}$ that contain an
even number of `a`s, while $r_2$ describes the strings that contain an even number of `b`s.

In [ ]:
let sigma = FSet.of_list [ 'a'; 'b' ]
let r1    = parse "(b*ab*a)*b*"
let r2    = parse "(a*ba*b)*a*"
;;
Printf.printf "size(r1) = %d, size(r2) = %d\n%!" (size r1) (size r2)

The minimized <span style="font-variant:small-caps;">Dfa</span>s for $r_1$ and $r_2$ have two states each, so the product automaton has four states.

In [ ]:
let f = intersection_dfa r1 r2 sigma
;;
FSet.cardinal f.states

In [ ]:
let r = intersection r1 r2 sigma
;;
size r

Although $r_1$ and $r_2$ have only 13 nodes each, the regular expression $r_1 \wedge r_2$ has 772 nodes.
Let us check that it is correct.

**Note:** The Python version of this notebook computes a regular expression with 784 nodes.  The function
`rpq` eliminates the states of the automaton one after the other, and the size of the result depends on the
order in which the states are eliminated.  The elements of an OCaml set of type `FSet.t` are sorted, while Python
stores the elements of a set in an order that depends on their hash values.  For the same reason, the simplified
regular expressions computed below differ from those of the Python version.  Of course, all of them are correct.

In [ ]:
accepts_same_language r f sigma

After simplification, the regular expression is much smaller, but it is still far from readable:

In [ ]:
let s = simplify r
;;
size s

In [ ]:
print_endline (to_str s)

In [ ]:
accepts_same_language s f sigma

A human would describe the intersection of these two languages differently.  For example, a string contains
both an even number of `a`s and an even number of `b`s iff it can be split into blocks of length two such that
the number of blocks of the form `ab` or `ba` is even.  This leads to the regular expression
$$ \bigl(\texttt{a} \cdot \texttt{a} + \texttt{b} \cdot \texttt{b} + (\texttt{a} \cdot \texttt{b} + \texttt{b} \cdot \texttt{a}) \cdot (\texttt{a} \cdot \texttt{a} + \texttt{b} \cdot \texttt{b})^* \cdot (\texttt{a} \cdot \texttt{b} + \texttt{b} \cdot \texttt{a})\bigr)^*, $$
which has only 33 nodes.

In [ ]:
let h = parse "(aa+bb+(ab+ba)(aa+bb)*(ab+ba))*"
;;
size h, accepts_same_language h f sigma

## Growth of the Regular Expressions

The function `dfa_2_regexp` calls the function `rpq`, which calls itself four times for every state of the
<span style="font-variant:small-caps;">Dfa</span>.  Therefore, the size of the regular expression computed for a <span style="font-variant:small-caps;">Dfa</span> with $n$ states grows like $4^n$.
As the number of states of the product automaton is the product of the numbers of states of $F_1$ and $F_2$,
the size of $r_1 \wedge r_2$ grows exponentially in this product.  The following table shows this for the
languages of strings in which the number of `a`s is divisible by $k$ and the number of `b`s is divisible by $m$.

**Function `divisible c d k`**
- *Input:* `c` and `d` are the two characters of the alphabet and `k` is a positive natural number.
- *Output:* A string that is parsed into a regular expression describing the strings over the alphabet
  $\{c, d\}$ in which the number of occurrences of `c` is divisible by `k`.

In [ ]:
let divisible (c : char) (d : char) (k : int) : string =
  "(" ^ String.concat "" (List.init k (fun _ -> Printf.sprintf "%c*%c" d c)) ^ Printf.sprintf ")*%c*" d

In [ ]:
print_endline (divisible 'a' 'b' 2);
print_endline (divisible 'b' 'a' 3)

In [ ]:
print_endline
  (String.concat " "
     [ rjust "k" 2; rjust "m" 2; rjust "states" 7; rjust "size(r1)" 9; rjust "size(r2)" 9;
       rjust "size(r1 ∧ r2)" 14; rjust "simplified" 11; rjust "correct" 8; rjust "time" 7 ]);
List.iter
  (fun (k, m) ->
    let r1    = parse (divisible 'a' 'b' k) in
    let r2    = parse (divisible 'b' 'a' m) in
    let start = Unix.gettimeofday () in
    let f     = intersection_dfa r1 r2 sigma in
    let r     = dfa_2_regexp f in
    let s     = simplify r in
    let stop  = Unix.gettimeofday () in
    let ok    = accepts_same_language s f sigma in
    Printf.printf "%2d %2d %7d %9d %9d %14d %11d %8b %6.2fs\n%!"
      k m (FSet.cardinal f.states) (size r1) (size r2) (size r) (size s) ok (stop -. start))
  [ (2, 2); (2, 3); (3, 3) ]

Although the regular expressions $r_1$ and $r_2$ are tiny, the regular expression $r_1 \wedge r_2$ has almost
800&nbsp;000 nodes when the product automaton has only nine states.  Simplification removes most of these nodes, but
the result is still much larger than necessary.  For a product automaton with twelve states the unsimplified regular
expression would have more than $4^{12} \approx 1.7 \cdot 10^7$ nodes.  This is why we do not implement the
function $\wedge$ in practice:  If we need the intersection of two regular languages, we work with the product
automaton instead.